# diag3 — `b_k` 의 무엇이 일을 하는가

diag0·diag1·diag2 가 확정한 것:

- 역방향 브랜치 = 관측 무관 상수표 `b_k`, `out_k = LN(0.5·(fwd_k + b_k))` **항등식**
- 순방향은 위치가 뭉개져 있고 (활용률 **0.27**, ACM2 는 **0.18**), `b_k` 가 갈라 준다 (**0.59**)
- `b_k` 를 끄면 위치별 오차가 **1.4~2.3배**

**남은 질문: `b_k` 의 어떤 성질이 일을 하는가?**
크기인가, 위치마다 다르다는 것인가, 아니면 학습된 특정 대응인가.

diag1 의 `compile_decoder` 가 임의의 표를 받으므로 **학습 없이** 잴 수 있다.
한 번 컴파일한 뒤 `dec._bk` 만 갈아 끼우므로 변형 하나당 forward 한 번이다.

## 비교하는 표

| 표 | 무엇 |
|---|---|
| `real` | `b` 그대로 — 기준 (회복률 100%) |
| `zero` | 0 — 하한 (회복률 0%) |
| **`mean`** | **모든 위치에 `b` 의 평균벡터 하나** ← 핵심 대조 |
| `shuffle` | 같은 벡터들, 위치만 섞음 |
| `random_ortho` | 랜덤 직교 행, `b` 의 위치별 norm 으로 맞춤 |
| `alpha=X` | `X·b` — 크기 스윕 |

### `mean` 이 제일 중요하다

크기와 공통 성분은 그대로 두고 **위치별 변화만** 없앤다. `zero` 를 0%, `real` 을 100% 로
둔 회복률로 읽는다.

- **`mean` 회복률이 낮다** → 이득은 거의 전부 "위치마다 다르다" 에서 온다.
  큰 상수를 더하는 것만으로는 설명되지 않는다 → **"위치 분리" 해석 확정**
- **`mean` 회복률이 높다** → 위치와 무관하게 큰 벡터를 더하는 것으로 대부분 설명된다
  → 해석을 다시 세워야 한다

### `shuffle` vs `random_ortho`

둘 다 모델이 학습 때 본 적 없는 표다. **OOD 조건이 같으므로 그 차이는 OOD 효과가
상쇄된 뒤 남는 것이다.**

- 둘이 비슷 → 내용보다 "위치가 갈라진다" 가 본질
- `shuffle` 이 확실히 낮음 → 학습된 벡터 *집합* 자체에 값어치가 있음
- 둘 다 `real` 보다 한참 높음 → **어느 위치에 어느 벡터가 가는지**도 중요

---

## ⚠️ 해석 주의

모든 변형은 **학습 때 없던 표를 추론에 꽂는 것**이라 그 자체로 OOD 다.
떨어지는 게 "OOD 라서" 인지 "그 성질이 중요해서" 인지 단독으로는 못 가른다.

> **변형 하나의 절대 수치는 읽지 말고, 변형끼리의 비교만 읽을 것.**

그래서 이 노트북은 **보조 증거**다. 주장의 무게는 diag2 의 활용률(0.28 → 0.59)이 진다.
그건 성능이 아니라 벡터에 대한 직접 계산이라 OOD 가 낄 자리가 없다.

---

> 커널 얘기는 diag0~2 와 같다. `lerobot` 을 커널로 import 하지 않고
> `mamba_ssm` 이 깔린 venv 로 subprocess 호출한 뒤 결과 json 만 읽는다.

## 0) 부팅

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

_h = Path.cwd()
REPO = next(c for c in (_h, *_h.parents)
            if (c / 'notebooks' / 'libero' / 'diag3_bk_variants.py').exists())
sys.path.insert(0, str(REPO / 'notebooks'))

try:
    import common_v23 as v23
    PYTHON = v23.PYTHON
except Exception as e:
    print('common_v23 import 실패, 기본값 사용:', e)
    PYTHON = os.environ.get('LEROBOT_PYTHON') or str(
        Path.home() / 'lerobot_project' / 'lerobot_env' / 'bin' / 'python')

SCRIPT = REPO / 'notebooks' / 'libero' / 'diag3_bk_variants.py'
SHARE = Path(os.environ.get('LEROBOT_OUTPUT',
                            Path.home() / 'lerobot_project' / 'outputs')) / 'final' / 'share' / 'diag3'
SHARE.mkdir(parents=True, exist_ok=True)

print('repo   :', REPO)
print('python :', PYTHON, '  (있음)' if Path(PYTHON).exists() else '  <- 없다!')
print('script :', SCRIPT, '  (있음)' if SCRIPT.exists() else '  <- 없다!')
print('share  :', SHARE)

## 1) 설정

In [ ]:
SEED   = 0
STEP   = 150_000      # -1 이면 최신 체크포인트
TASK   = 'libero_10'
BATCH  = 4
GPU    = '0'
STAMP  = time.strftime('%Y%m%d_%H%M')

ENV = dict(os.environ,
           PYTHONPATH=str(REPO / 'src'),
           HF_HUB_DISABLE_XET='1',
           MPLBACKEND='Agg',
           CUDA_VISIBLE_DEVICES=GPU)

def run_diag3(tags, json_path):
    cmd = [PYTHON, str(SCRIPT), '--tags', tags, '--seed', str(SEED), '--step', str(STEP),
           '--task', TASK, '--batch', str(BATCH), '--json', str(json_path)]
    print('$', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=ENV, cwd=str(REPO))
    for line in p.stdout:
        print(line, end='')
    print(f'\n[exit {p.wait()}]')
    return json.loads(Path(json_path).read_text(encoding='utf-8')) if Path(json_path).exists() else {}

## 2) K=100 단일 — 이 셀이 답이다

`mean` 의 회복률만 보면 된다. **낮으면 "위치 분리" 해석이 확정된다.**

In [ ]:
res100 = run_diag3('bimamba_pure', SHARE / f'diag3_k100_{STAMP}.json')

## 3) K 50 / 100 / 150 전부

In [ ]:
results = run_diag3('all', SHARE / f'diag3_all_{STAMP}.json')
print('\n받은 태그:', list(results))

## 4) 요약표 — 회복률

In [ ]:
import csv

K_OF = {'bimamba_pure_k50': 50, 'bimamba_pure': 100, 'bimamba_pure_k150': 150}
order = [(t, r) for t, r in results.items() if r.get('ok')]
order.sort(key=lambda kv: K_OF.get(kv[0], 0))

NAMED = ('real', 'zero', 'mean', 'shuffle', 'random_ortho')
hdr = f"{'K':>4} {'표':<14} {'활용률':>8} {'오차':>9} {'회복률':>8}"
print(hdr); print('-' * len(hdr))
rows = []
for tag, r in order:
    v = r['variants']
    span = v['zero']['err_mean'] - v['real']['err_mean']
    keys = [k for k in NAMED if k in v] + sorted(
        (k for k in v if k.startswith('alpha=')), key=lambda k: float(k.split('=')[1]))
    for name in keys:
        d = v[name]
        rec = (v['zero']['err_mean'] - d['err_mean']) / span if span > 1e-9 else float('nan')
        print(f"{r['K']:>4} {name:<14} {d['use_ratio']:>8.2f} {d['err_mean']:>9.4f} {rec:>7.0%}")
        rows.append({'K': r['K'], 'tag': tag, 'table': name, 'use_ratio': d['use_ratio'],
                     'cos_adj_mean': d['cos_adj_mean'], 'err_mean': d['err_mean'],
                     'recovery': rec})
    print('-' * len(hdr))

if rows:
    csv_path = SHARE / f'diag3_summary_{STAMP}.csv'
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print('saved', csv_path)

## 5) 핵심 그림 — 무엇이 이득을 만드는가

`zero` 를 0%, `real` 을 100% 로 둔 회복률. **`mean` 막대가 낮을수록
"위치마다 다르다" 가 이득의 원천이라는 뜻이다.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

show = [k for k in ('zero', 'mean', 'random_ortho', 'shuffle', 'real') ]
label = {'zero': 'b_k = 0', 'mean': 'same vector\nevery position',
         'random_ortho': 'random\northogonal', 'shuffle': 'shuffled\npositions',
         'real': 'learned b_k'}

fig, ax = plt.subplots(figsize=(7.4, 4.2))
w = 0.8 / max(len(order), 1)
for j, (tag, r) in enumerate(order):
    v = r['variants']
    span = v['zero']['err_mean'] - v['real']['err_mean']
    ys = [(v['zero']['err_mean'] - v[k]['err_mean']) / span if (k in v and span > 1e-9)
          else np.nan for k in show]
    ax.bar(np.arange(len(show)) + j * w - 0.4 + w / 2, ys, w,
           label=f"K={r['K']}")
ax.set_xticks(range(len(show))); ax.set_xticklabels([label[k] for k in show], fontsize=9)
ax.set_ylabel('error recovery  (b_k=0 is 0%, learned b_k is 100%)')
ax.set_title('Which property of the table does the work?')
ax.axhline(0, color='k', lw=.6); ax.axhline(1, color='k', lw=.6, ls=':')
ax.grid(alpha=.3, axis='y'); ax.legend()
fig.tight_layout()
png = SHARE / f'diag3_recovery_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight')
print('saved', png)
plt.show()

## 6) 크기 스윕 — `α·b_k`

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for tag, r in order:
    v = r['variants']
    pts = [(0.0, v['zero']['err_mean']), (1.0, v['real']['err_mean'])]
    pts += [(float(k.split('=')[1]), v[k]['err_mean']) for k in v if k.startswith('alpha=')]
    pts.sort()
    ax.plot([p[0] for p in pts], [p[1] for p in pts], 'o-', lw=1.6, label=f"K={r['K']}")
ax.axvline(1.0, color='k', lw=.8, ls=':')
ax.set_xlabel('scale  alpha  (b_k -> alpha * b_k)'); ax.set_ylabel('mean |pred - target|')
ax.set_title('Is the learned magnitude tuned?')
ax.grid(alpha=.3); ax.legend()
fig.tight_layout()
png2 = SHARE / f'diag3_alpha_{STAMP}.png'
fig.savefig(png2, dpi=150, bbox_inches='tight')
print('saved', png2)
plt.show()

## 7) 결과 읽는 법

### `mean` 회복률 (셀 4·5) — 이 노트북의 답

| 회복률 | 뜻 |
|---|---|
| **< 25%** | 이득은 거의 전부 **위치별 변화**에서 온다. diag2 의 "위치 분리" 해석 확정 |
| 25~75% | 위치별 변화와 전역 성분이 둘 다 기여 |
| **> 75%** | 위치와 무관하게 큰 벡터를 더하는 것으로 설명된다. **해석을 다시 세울 것** |

### `shuffle` vs `random_ortho` (셀 4·5)

둘 다 OOD 조건이 같으므로 **차이만** 읽는다.

- 비슷 → 내용보다 "갈라진다" 가 본질
- `shuffle` 이 낮음 → 학습된 벡터 집합에 값어치가 있음
- **둘 다 `real` 보다 한참 높음** → 어느 위치에 어느 벡터가 가는지도 중요
  (= 단순 positional encoding 이 아니라 순방향과 짝이 맞는 표라는 뜻)

### `α` 스윕 (셀 6)

오차 최소가 `α=1` 이면 학습된 크기가 맞춰진 값이다. 최소가 다른 데 있으면
크기 자체는 덜 민감하다는 뜻이고, 그건 "분리만 되면 된다" 쪽 증거다.

### 논문에 어떻게 쓰나

**주장의 무게는 diag2 가 진다** (활용률 0.28 → 0.59 — 성능이 아니라 벡터 계산이라
OOD 무관). diag3 은 *"그 분리가 이득의 원천인가"* 를 보강하는 보조 증거다.

한 문장으로 쓸 수 있는 형태:

> Replacing $b_k$ with a single vector repeated at every position recovers only
> [[X]]\% of the error difference, indicating that the position-dependent variation
> rather than the added magnitude accounts for the effect.

### 한계

- seed 0 / step 150k / libero_10 / batch 4 단일 세팅
- **모든 변형이 OOD** — 절대 수치 금지, 비교만
- 성공률이 아니라 예측 오차다. 성공률로 확인하려면 eval 을 돌려야 한다